# Figure 2d — human microsome heatmap

Run all cells from this folder or the `P450_analysis/` root. The four direct inputs are in `input/`: `rxn_annotated_DnCl_features.tsv`, `rxn_annotated_DnHz_features.tsv`, and the corresponding two `Human_microsome_full_Feature_table_*_10ppm_min_peak_height_1e6.tsv` tables.

An rxn candidate is identified by steroid, derivatization, and its original `feature_id_pellets` / `feature_id_supernatant` fields. These fraction IDs are provenance, not Human microsome join keys. Matching uses m/z within ±10 ppm and retention time within ±5 s; every qualifying hit is recorded. Rank 1 is chosen by absolute RT error, then absolute ppm error, then Human microsome ID. **Signal does not affect the hit ranking.** For each steroid, top-500 TIC normalization of the nine samples precedes `log2(normalized abundance + 1)`, three-replicate means at 0/30/60 min, and row z-scores.

`results/` has three tables: `01_candidate_matches.tsv` (all 849 hit/unmatched rows for 833 candidates), `02_plotted_timecourse.tsv` (81 plotted columns × 3 time points = 243 rows), and `03_match_summary.tsv` (five steroids). The 81 plotted columns are the best m/z/RT hits with a positive raw signal in at least one sample. `plot_position` preserves each drawn column; Human microsome IDs can repeat, so do not deduplicate on that ID. `rxn_input_id` is the source table's original `id` used for the x-axis, not a generated matching key. The labeled and text-free panels are saved as PDF/SVG with their established layout.

## 1. Read inputs and set parameters

In [1]:
import ast
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.font_manager import FontProperties, fontManager
from matplotlib.patches import Rectangle

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "input" / "rxn_annotated_DnCl_features.tsv").is_file():
    PROJECT_DIR = PROJECT_DIR / "Figure_modification" / "Figure_2d_human_microsome"
INPUT_DIR = PROJECT_DIR / "input"
RESULTS = PROJECT_DIR / "results"
required_inputs = [
    INPUT_DIR / "rxn_annotated_DnCl_features.tsv",
    INPUT_DIR / "rxn_annotated_DnHz_features.tsv",
    INPUT_DIR / "Human_microsome_full_Feature_table_DnCl_10ppm_min_peak_height_1e6.tsv",
    INPUT_DIR / "Human_microsome_full_Feature_table_DnHz_10ppm_min_peak_height_1e6.tsv",
]
missing = [path for path in required_inputs if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing Figure 2d inputs: {missing}")

STEROID_CONFIGS = (
    ("Estradiol", "DnCl"), ("Estrone", "DnCl"),
    ("DHEA", "DnHz"), ("Pregnenolone", "DnHz"),
    ("Progesterone", "DnHz"),
)
STEROID_ORDER = [name for name, _ in STEROID_CONFIGS]
STEROID_INFO = dict(STEROID_CONFIGS)
KEY_COLUMNS = ["parent_id", "derivatization", "feature_id_pellets", "feature_id_supernatant"]
TIME_POINTS = (0, 30, 60)
MZ_TOLERANCE_PPM, RT_TOLERANCE_SECONDS = 10.0, 5.0

## 2. Match candidates and calculate profiles

In [2]:
def reaction_memberships(row: pd.Series) -> dict[str, dict[str, set[str]]]:
    """Return steroid and enzyme memberships from the two reaction fields."""
    memberships: dict[str, dict[str, set[str]]] = {}
    for fraction in ("pellets", "supernatant"):
        value = row[f"bioreactions_{fraction}"]
        if value in ("", "NA"):
            continue
        try:
            parsed = ast.literal_eval(value)
        except (SyntaxError, ValueError) as exc:
            raise ValueError(f"Cannot parse {fraction} reactions for {row['id']}") from exc

        def walk(item: object) -> None:
            if (
                isinstance(item, tuple)
                and len(item) >= 2
                and isinstance(item[0], str)
                and isinstance(item[1], str)
                and item[1].lower() in {name.lower() for name in STEROID_ORDER}
            ):
                steroid = next(name for name in STEROID_ORDER if name.lower() == item[1].lower())
                memberships.setdefault(steroid, {"pellets": set(), "supernatant": set()})
                memberships[steroid][fraction].add(item[0])
            elif isinstance(item, (list, tuple)):
                for child in item:
                    walk(child)

        walk(parsed)
    return memberships


def load_rxn_assignments() -> pd.DataFrame:
    records = []
    for deriv in ("DnCl", "DnHz"):
        source_file = INPUT_DIR / f"rxn_annotated_{deriv}_features.tsv"
        source = pd.read_csv(source_file, sep="\t", dtype=str, keep_default_na=False)
        required = {"id", "mz", "rtime", "source", "ion_relation",
                    "bioreactions_pellets", "bioreactions_supernatant",
                    "feature_id_pellets", "feature_id_supernatant"}
        missing = required - set(source.columns)
        if missing:
            raise ValueError(f"{source_file.name}: missing {sorted(missing)}")
        for source_row_number, row in enumerate(source.to_dict("records"), start=1):
            memberships = reaction_memberships(pd.Series(row))
            if not memberships:
                raise ValueError(f"{source_file.name} data row {source_row_number}: no steroid")
            for steroid, fractions in memberships.items():
                expected_deriv = STEROID_INFO[steroid]
                if deriv != expected_deriv:
                    raise ValueError(f"{source_file.name} row {source_row_number}: {steroid} is {expected_deriv}")
                record = dict(row)
                record.update({
                    "parent_id": steroid,
                    "derivatization": deriv,
                    "source_table": source_file.name,
                    "source_row_number": source_row_number,
                    "rxn_enzymes_pellets": ";".join(sorted(fractions["pellets"])),
                    "rxn_enzymes_supernatant": ";".join(sorted(fractions["supernatant"])),
                })
                records.append(record)

    candidates = pd.DataFrame(records)
    if candidates.duplicated(KEY_COLUMNS).any():
        raise ValueError("Steroid and original fraction feature IDs must identify one rxn row.")
    candidates["mz_numeric"] = pd.to_numeric(candidates["mz"], errors="raise")
    candidates["rtime_numeric"] = pd.to_numeric(candidates["rtime"], errors="raise")
    candidates["steroid_order"] = candidates["parent_id"].map(
        {name: i for i, name in enumerate(STEROID_ORDER)}
    )
    candidates = candidates.sort_values(
        ["steroid_order", "mz_numeric", "rtime_numeric", "source_row_number"],
        kind="stable",
    ).reset_index(drop=True)

    return candidates.drop(columns=["mz_numeric", "rtime_numeric", "steroid_order"])


def timecourse_sample_columns(feature_table: pd.DataFrame, steroid: str) -> dict[int, list[str]]:
    selected = {}
    for time in TIME_POINTS:
        pattern = re.compile(
            rf"^Human_microsome_{re.escape(steroid)}_{time}min_(\d+)(?:_|$)",
            flags=re.IGNORECASE,
        )
        matches = []
        for column in feature_table.columns[11:]:
            match = pattern.search(column)
            if match:
                matches.append((int(match.group(1)), column))
        matches.sort()
        if [rep for rep, _ in matches] != [1, 2, 3]:
            raise ValueError(f"{steroid} {time} min: expected replicates 1, 2, 3; got {matches}")
        selected[time] = [column for _, column in matches]
    return selected


def normalize_top500_then_log2(raw_data: pd.DataFrame) -> pd.DataFrame:
    top_features = raw_data.mean(axis=1).nlargest(min(500, len(raw_data))).index
    top500_tic = raw_data.loc[top_features].sum(axis=0)
    if (top500_tic <= 0).any():
        raise ValueError("Top-500 TIC must be positive in every sample.")
    sample_scale = top500_tic / top500_tic.mean()
    return np.log2(raw_data.div(sample_scale, axis=1) + 1)


def row_zscore(values: np.ndarray) -> tuple[np.ndarray, bool]:
    standard_deviation = values.std(ddof=0)
    if not np.isfinite(standard_deviation) or standard_deviation == 0:
        return np.zeros(values.shape, dtype=float), False
    return (values - values.mean()) / standard_deviation, True


def candidate_hits(candidate: pd.Series, feature_table: pd.DataFrame) -> list[dict]:
    mz = float(candidate["mz"])
    rt = float(candidate["rtime"])
    mz_values = feature_table["mz"].to_numpy(dtype=float)
    rt_values = feature_table["rtime"].to_numpy(dtype=float)
    ppm_errors = np.abs(mz_values - mz) / mz * 1e6
    rt_differences = np.abs(rt_values - rt)
    positions = np.flatnonzero(
        (ppm_errors <= MZ_TOLERANCE_PPM) & (rt_differences <= RT_TOLERANCE_SECONDS)
    )
    hits = [{
        "row_position": int(pos),
        "human_microsome_feature": str(feature_table.iloc[pos]["id_number"]),
        "human_microsome_mz": float(mz_values[pos]),
        "human_microsome_rtime": float(rt_values[pos]),
        "mz_error_ppm": float(ppm_errors[pos]),
        "rt_difference_seconds": float(rt_differences[pos]),
    } for pos in positions]
    return sorted(hits, key=lambda h: (
        h["rt_difference_seconds"], h["mz_error_ppm"], h["human_microsome_feature"]
    ))


def match_and_normalize(candidates: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    feature_tables = {}
    for deriv in ("DnCl", "DnHz"):
        path = INPUT_DIR / f"Human_microsome_full_Feature_table_{deriv}_10ppm_min_peak_height_1e6.tsv"
        table = pd.read_csv(path, sep="\t", low_memory=False, dtype={"id_number": str})
        required = {"id_number", "mz", "rtime"}
        if required - set(table.columns):
            raise ValueError(f"{path.name}: missing {sorted(required - set(table.columns))}")
        if table["id_number"].duplicated().any():
            raise ValueError(f"{path.name}: duplicate human-microsome feature ID")
        feature_tables[deriv] = table

    records = []
    all_hits = []
    for steroid, deriv in STEROID_CONFIGS:
        feature_table = feature_tables[deriv]
        sample_groups = timecourse_sample_columns(feature_table, steroid)
        ordered_samples = [col for time in TIME_POINTS for col in sample_groups[time]]
        raw_data = feature_table.set_index("id_number")[ordered_samples].astype(float)
        normalized_log2 = normalize_top500_then_log2(raw_data)
        steroid_candidates = candidates.loc[candidates["parent_id"].eq(steroid)]
        for _, candidate in steroid_candidates.iterrows():
            record = candidate.to_dict()
            hits = candidate_hits(candidate, feature_table)
            for rank, hit in enumerate(hits, start=1):
                hid = hit["human_microsome_feature"]
                all_hits.append({
                    "parent_id": steroid,
                    "derivatization": deriv,
                    "feature_id_pellets": candidate["feature_id_pellets"],
                    "feature_id_supernatant": candidate["feature_id_supernatant"],
                    "source_row_number": candidate["source_row_number"],
                    "id": candidate["id"],
                    "match_rank": rank,
                    "is_selected_best": rank == 1,
                    **{key: value for key, value in hit.items() if key != "row_position"},
                    "raw_detection_count_across_9_samples": int((raw_data.loc[hid] > 0).sum()),
                })
            record["number_of_matches_within_tolerance"] = len(hits)
            if not hits:
                record.update({
                    "match_status": "unmatched",
                    "human_microsome_feature": pd.NA,
                    "human_microsome_mz": np.nan,
                    "human_microsome_rtime": np.nan,
                    "mz_error_ppm": np.nan,
                    "rt_difference_seconds": np.nan,
                    "raw_detection_count_across_9_samples": 0,
                })
                for time in TIME_POINTS:
                    record[f"mean_normalized_log2_{time}min"] = np.nan
                    record[f"zscore_{time}min"] = 0.0
            else:
                best = {key: value for key, value in hits[0].items() if key != "row_position"}
                hid = best["human_microsome_feature"]
                means = np.array([
                    normalized_log2.loc[hid, sample_groups[time]].mean()
                    for time in TIME_POINTS
                ], dtype=float)
                zscores, has_variation = row_zscore(means)
                record.update(best)
                record["match_status"] = "matched_zscore" if has_variation else "matched_constant_zero"
                record["raw_detection_count_across_9_samples"] = int((raw_data.loc[hid] > 0).sum())
                for time, mean, zscore in zip(TIME_POINTS, means, zscores):
                    record[f"mean_normalized_log2_{time}min"] = float(mean)
                    record[f"zscore_{time}min"] = float(zscore)
            records.append(record)
    return pd.DataFrame(records), pd.DataFrame(all_hits)

In [3]:
RESULTS.mkdir(parents=True, exist_ok=True)
candidates = load_rxn_assignments()
details, all_hits = match_and_normalize(candidates)
if len(details) != 833 or details.duplicated(KEY_COLUMNS).any():
    raise ValueError("Expected 833 distinct steroid/fraction-ID combinations")

summary_rows = []
for steroid in STEROID_ORDER:
    part = details.loc[details.parent_id.eq(steroid)]
    matched = part.match_status.ne("unmatched")
    with_signal = matched & part.raw_detection_count_across_9_samples.gt(0)
    summary_rows.append({
        "steroid": steroid,
        "rxn_steroid_features": len(part),
        "matched_mz_rt": int(matched.sum()),
        "matched_with_signal": int(with_signal.sum()),
        "matched_all_nine_samples_zero": int((matched & ~with_signal).sum()),
        "unmatched": int((~matched).sum()),
        "candidates_with_multiple_human_hits": int(part.number_of_matches_within_tolerance.gt(1).sum()),
    })
summary = pd.DataFrame(summary_rows)


def timecourse_long(rows):
    """Keep original fraction IDs as columns instead of inventing a matrix ID."""
    source = ["plot_position", *KEY_COLUMNS, "id", "human_microsome_feature",
              "raw_detection_count_across_9_samples"]
    long = rows[source].iloc[np.repeat(np.arange(len(rows)), len(TIME_POINTS))].reset_index(drop=True)
    long["time_min"] = np.tile(TIME_POINTS, len(rows))
    long["mean_normalized_log2"] = rows[
        [f"mean_normalized_log2_{time}min" for time in TIME_POINTS]
    ].to_numpy().reshape(-1)
    long["row_zscore"] = rows[
        [f"zscore_{time}min" for time in TIME_POINTS]
    ].to_numpy().reshape(-1)
    return long.rename(columns={
        "parent_id": "steroid", "id": "rxn_input_id",
        "human_microsome_feature": "human_microsome_feature_id",
    })


candidate_columns = [*KEY_COLUMNS, "mz", "rtime", "number_of_matches_within_tolerance"]
hit_columns = [
    *KEY_COLUMNS, "match_rank", "human_microsome_feature", "human_microsome_mz",
    "human_microsome_rtime", "mz_error_ppm", "rt_difference_seconds",
    "raw_detection_count_across_9_samples",
]
matches = details[candidate_columns].assign(_candidate_order=np.arange(len(details))).merge(
    all_hits[hit_columns], on=KEY_COLUMNS, how="left", sort=False,
).sort_values(["_candidate_order", "match_rank"], kind="stable")
matches = matches.rename(columns={
    "parent_id": "steroid", "mz": "source_mz", "rtime": "source_rt_s",
    "number_of_matches_within_tolerance": "match_count",
    "human_microsome_feature": "human_microsome_feature_id",
    "human_microsome_rtime": "human_microsome_rt_s",
    "mz_error_ppm": "abs_mz_error_ppm",
    "rt_difference_seconds": "abs_rt_difference_s",
})
matches["match_rank"] = matches.match_rank.astype("Int64")
matches["match_status"] = np.where(
    matches.match_count.eq(0), "unmatched",
    np.where(matches.match_count.eq(1), "unique", "ambiguous"),
)
matches["raw_detection_count_across_9_samples"] = (
    matches.raw_detection_count_across_9_samples.fillna(0).astype(int)
)
match_columns = [
    "steroid", "derivatization", "feature_id_pellets", "feature_id_supernatant",
    "source_mz", "source_rt_s", "human_microsome_feature_id",
    "human_microsome_mz", "human_microsome_rt_s", "abs_mz_error_ppm",
    "abs_rt_difference_s", "match_count", "match_rank", "match_status",
    "raw_detection_count_across_9_samples",
]
matches = matches[match_columns].reset_index(drop=True)
candidate_key = ["steroid", "derivatization", "feature_id_pellets", "feature_id_supernatant"]
if (len(matches), matches[candidate_key].drop_duplicates().shape[0]) != (849, 833):
    raise AssertionError("Unexpected candidate–hit table size")
if matches.match_status.value_counts().to_dict() != {
    "unmatched": 704, "unique": 113, "ambiguous": 32,
}:
    raise AssertionError("Unexpected candidate–hit status counts")
best_hits = matches.loc[matches.match_rank.eq(1)]
if (len(best_hits), int(best_hits.raw_detection_count_across_9_samples.gt(0).sum())) != (129, 81):
    raise AssertionError("Best-hit counts differ from heatmap inputs")

print(f"{matches[candidate_key].drop_duplicates().shape[0]} candidate records; {len(matches)} candidate–hit rows")

833 candidate records; 849 candidate–hit rows


## 3. Draw labeled and text-free heatmaps

In [4]:
from contextlib import nullcontext

ZSCORE_COLUMNS = [f'zscore_{time}min' for time in TIME_POINTS]
OUTPUT_STEM = 'human_microsome_rxn_annotated_matched_with_signal_170mm'


def select_plot_features(details):
    selected = details.loc[
        details['match_status'].ne('unmatched')
        & details['raw_detection_count_across_9_samples'].gt(0)
    ].copy()
    unknown = set(selected['parent_id']) - set(STEROID_ORDER)
    if unknown:
        raise ValueError(f'Unknown steroid groups: {sorted(unknown)}')
    selected = pd.concat([
        selected.loc[selected['parent_id'].eq(steroid)]
        for steroid in STEROID_ORDER
    ], ignore_index=True)
    if selected.empty or selected.duplicated(KEY_COLUMNS).any():
        raise ValueError('Plot requires unique steroid/fraction-ID combinations.')
    if selected.duplicated(['parent_id', 'id']).any():
        raise ValueError('Original input IDs must be unique within each plotted steroid block.')
    if not np.isfinite(selected[ZSCORE_COLUMNS].to_numpy(dtype=float)).all():
        raise ValueError('Selected z-scores must be finite.')
    return selected


def draw_heatmap(details, labels=True):
    """Draw the labeled or text-free 170 mm panel from the same selected rows."""
    selected = select_plot_features(details)
    RESULTS.mkdir(parents=True, exist_ok=True)
    arial_path = Path('/System/Library/Fonts/Supplemental/Arial.ttf')
    if labels:
        if not arial_path.is_file():
            raise FileNotFoundError('Arial.ttf is required for the requested font.')
        fontManager.addfont(str(arial_path))
        arial = FontProperties(family='Arial', fname=str(arial_path), size=8)

    page_width, page_height = 170.0, 46.0 if labels else 28.5
    plot_left, plot_right = (10.0 if labels else 4.0), 2.5
    plot_width = page_width - plot_left - plot_right
    tile_size, tile_height = 1.0, 1.0
    feature_gap, time_gap, steroid_gap = 0.20, 1.2, 1.15
    frame_padding, frame_color = 0.25, '#999999'
    style = {
        'font.family': 'Arial', 'font.size': 8,
        'xtick.labelsize': 7, 'ytick.labelsize': 8,
        'axes.labelsize': 8, 'axes.titlesize': 9,
        'pdf.fonttype': 42, 'ps.fonttype': 42,
        'svg.fonttype': 'none', 'axes.unicode_minus': True,
    }

    # The no-text version deliberately uses the default context, as before.
    with plt.rc_context(style) if labels else nullcontext():
        fig = plt.figure(figsize=(page_width / 25.4, page_height / 25.4), facecolor='white')
        color_map = sns.color_palette('vlag', as_cmap=True)
        normalization = Normalize(vmin=-1.5, vmax=1.5)
        values = selected[ZSCORE_COLUMNS].to_numpy(dtype=float).T

        x_centers, group_spans = [], []
        cursor = 0.0
        for steroid in STEROID_ORDER:
            count = int(selected['parent_id'].eq(steroid).sum())
            positions = cursor + np.arange(count) * (tile_size + feature_gap)
            start, end = float(positions[0]), float(positions[-1] + tile_size)
            x_centers.extend(positions + tile_size / 2)
            group_spans.append((steroid, start, end))
            cursor = end + feature_gap + steroid_gap

        matrix_width = group_spans[-1][2]
        matrix_height = 3 * tile_height + 2 * time_gap
        plot_height = plot_width * (matrix_height + 2 * frame_padding) / (matrix_width + 2 * frame_padding)
        plot_bottom = 20.2 if labels else 2.4
        axis = fig.add_axes([
            plot_left / page_width, plot_bottom / page_height,
            plot_width / page_width, plot_height / page_height,
        ])
        y_centers = np.arange(3) * (tile_height + time_gap) + tile_height / 2

        for column, x_center in enumerate(x_centers):
            for row, y_center in enumerate(y_centers):
                axis.add_patch(Rectangle(
                    (x_center - tile_size / 2, y_center - tile_height / 2),
                    tile_size, tile_height,
                    facecolor=color_map(normalization(values[row, column])),
                    edgecolor='none', linewidth=0, zorder=2,
                ))
        for previous, following in zip(group_spans[:-1], group_spans[1:]):
            axis.axvline(
                (previous[2] + following[1]) / 2,
                color='#AAAAAA', linewidth=0.65,
                linestyle=(0, (2.0, 2.5)), zorder=1,
            )

        axis.set_xlim(-frame_padding, matrix_width + frame_padding)
        axis.set_ylim(matrix_height + frame_padding, -frame_padding)
        axis.set_aspect('equal', adjustable='box')
        if labels:
            axis.set_xticks(x_centers, labels=selected['id'].tolist())
            plt.setp(
                axis.get_xticklabels(), rotation=80, ha='right', va='top',
                rotation_mode='anchor',
                fontproperties=FontProperties(family='Arial', fname=str(arial_path), size=7),
            )
            axis.set_yticks(y_centers, labels=[f'{time} min' for time in TIME_POINTS])
            plt.setp(axis.get_yticklabels(), fontproperties=arial)
            axis.tick_params(axis='x', bottom=True, top=False, length=2.3, width=0.55,
                             pad=2, color=frame_color, labelcolor='black')
            axis.tick_params(axis='y', left=True, right=False, length=2.3, width=0.55,
                             pad=3, color=frame_color, labelcolor='black')
        else:
            axis.set_xticks([])
            axis.set_yticks(y_centers, labels=['', '', ''])
            axis.tick_params(axis='y', left=True, right=False, direction='out',
                             length=2.3, width=0.55, pad=0, color=frame_color)
        for spine in axis.spines.values():
            spine.set_visible(True)
            spine.set_color(frame_color)
            spine.set_linewidth(0.65)

        if labels:
            group_font = FontProperties(family='Arial', fname=str(arial_path), size=9)
            group_titles = [axis.annotate(
                steroid, xy=((start + end) / 2, 1),
                xycoords=axis.get_xaxis_transform(), xytext=(0, 3.5),
                textcoords='offset points', ha='center', va='bottom',
                fontproperties=group_font, color='#377DA5', annotation_clip=False,
            ) for steroid, start, end in group_spans]

        colorbar_width, colorbar_height = 16.0, 1.2
        colorbar_axis = fig.add_axes([
            (plot_left + plot_width - 8.0 - colorbar_width) / page_width,
            (40.0 if labels else 22.5) / page_height,
            colorbar_width / page_width, colorbar_height / page_height,
        ])
        colorbar = fig.colorbar(
            ScalarMappable(norm=normalization, cmap=color_map),
            cax=colorbar_axis, orientation='horizontal',
            ticks=[-1.5, 0, 1.5] if labels else [],
        )
        if labels:
            colorbar.set_label('Row z-score', fontproperties=arial, labelpad=2)
            colorbar.ax.xaxis.set_label_position('top')
            colorbar.ax.tick_params(labelsize=8, length=2, width=0.5, pad=2)
            plt.setp(colorbar.ax.get_xticklabels(), fontproperties=arial)
        else:
            colorbar.ax.tick_params(bottom=False, top=False, labelbottom=False, labeltop=False)
        colorbar.outline.set_edgecolor(frame_color)
        colorbar.outline.set_linewidth(0.5)

        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        assert len(axis.patches) == 3 * len(selected)
        tile_box = axis.patches[0].get_window_extent(renderer)
        assert abs(tile_box.height / tile_box.width - tile_height / tile_size) < 1e-6
        if labels:
            label_boxes = [label.get_window_extent(renderer) for label in axis.get_xticklabels()]
            assert all(label.get_rotation() == 80 and label.get_fontsize() == 7
                       for label in axis.get_xticklabels())
            assert min(box.y0 for box in label_boxes) > fig.bbox.y0
            time_boxes = sorted(
                (label.get_window_extent(renderer) for label in axis.get_yticklabels()),
                key=lambda box: box.y0,
            )
            assert all(lower.y1 + 2 < upper.y0 for lower, upper in zip(time_boxes[:-1], time_boxes[1:]))
            title_boxes = [label.get_window_extent(renderer) for label in group_titles]
            legend_boxes = [label.get_window_extent(renderer)
                            for label in [colorbar.ax.xaxis.label, *colorbar.ax.get_xticklabels()]]
            tick_boxes = [label.get_window_extent(renderer) for label in colorbar.ax.get_xticklabels()]
            assert all(left.x1 + 1 < right.x0 for left, right in zip(tick_boxes[:-1], tick_boxes[1:]))
            assert min(box.y0 for box in legend_boxes) > max(box.y1 for box in title_boxes) + 2
            assert max(box.y1 for box in legend_boxes) < fig.bbox.y1
            assert all(label.get_window_extent(renderer).y0 > label.axes.get_window_extent(renderer).y1
                       for label in group_titles)
        else:
            assert not axis.get_xticks().size
            assert axis.get_yticks().size == 3
            assert not any(text.get_text() for text in fig.findobj(plt.Text))

        stem = OUTPUT_STEM if labels else f'{OUTPUT_STEM}_no_text'
        for suffix in ('pdf', 'svg'):
            fig.savefig(RESULTS / f'{stem}.{suffix}', dpi=300, facecolor='white')

    return fig, axis, selected


labeled_figure, _, plotted = draw_heatmap(details, labels=True)
no_text_figure, _, no_text_plotted = draw_heatmap(details, labels=False)
plt.close(labeled_figure)
plt.close(no_text_figure)
if len(plotted) != 81 or len(no_text_plotted) != 81:
    raise ValueError("Expected 81 plotted features")
print(summary.to_string(index=False))
print(f"Plotted {len(plotted)} features; wrote four PDF/SVG files to {RESULTS}")

     steroid  rxn_steroid_features  matched_mz_rt  matched_with_signal  matched_all_nine_samples_zero  unmatched  candidates_with_multiple_human_hits
   Estradiol                   245             25                   13                             12        220                                    3
     Estrone                   163              8                    8                              0        155                                    2
        DHEA                   163             30                   30                              0        133                                    2
Pregnenolone                   182             43                   11                             32        139                                    4
Progesterone                    80             23                   19                              4         57                                    5
Plotted 81 features; wrote four PDF/SVG files to /Users/zhengjie/Desktop/P450_analysis/Figure_modifi

## 4. Export three concise tables

In [5]:
# Export only the match audit, plotted feature–time data, and steroid summary.
matches.to_csv(RESULTS / "01_candidate_matches.tsv", sep="\t", index=False)

plotted_with_position = plotted.copy()
plotted_with_position.insert(0, "plot_position", np.arange(1, len(plotted) + 1))
plotted_timecourse = timecourse_long(plotted_with_position)
plot_columns = [
    "plot_position", "steroid", "derivatization", "feature_id_pellets",
    "feature_id_supernatant", "rxn_input_id", "human_microsome_feature_id",
    "raw_detection_count_across_9_samples", "time_min",
    "mean_normalized_log2", "row_zscore",
]
plotted_timecourse = plotted_timecourse[plot_columns]
if len(plotted_timecourse) != 243 or not plotted_timecourse.groupby("plot_position").size().eq(3).all():
    raise AssertionError("Expected three time points for each of 81 plotted columns")
if plotted_timecourse.plot_position.nunique() != 81:
    raise AssertionError("Plotted columns were lost or deduplicated")
plotted_timecourse.to_csv(RESULTS / "02_plotted_timecourse.tsv", sep="\t", index=False)

summary.to_csv(RESULTS / "03_match_summary.tsv", sep="\t", index=False)
print(f"Wrote 3 TSVs: {len(matches)} match rows, {len(plotted_timecourse)} plotted time rows, {len(summary)} steroids")


Wrote 3 TSVs: 849 match rows, 243 plotted time rows, 5 steroids
